# Step 0 (optional): is BreastDCEDL downloadable? Download, verify, look inside (CPU, ~5 min)
**Self-contained:** no repo and no GitHub secret. Session options: Accelerator **None**, Internet **ON**.

What it shows:
1. The data is open on Zenodo (record 18114231, CC BY-NC 4.0, no login). It downloads with parallel byte ranges,
   and each file is checked against its published md5. Zenodo sometimes returns short or `504` responses, so
   parts are size-checked and resumed.
2. What one patient looks like: one 3D NIfTI per DCE phase, raw float64 scanner values, and the tumour mask.
3. How a 3D scan becomes the 2D RGB slices the ViT sees, and why the original joint 99.5th-percentile clipping
   saturates the tumour.

`WHAT = "demo"` (55 MB, a 7-patient sample) is enough to answer the question. **This is not the training data:**
the full dataset is downloaded by `step1a/1b/1c_download_*.ipynb`.

In [ ]:
WHAT = "demo"      # "demo" (55 MB) | "ispy1" (1.25 GB) | "ispy2" (13.6 GB) | "duke" (8.6 GB)
PARTS = 8          # parallel connections
PID = None         # a patient to inspect, or None = first I-SPY patient found

In [ ]:
import hashlib, os, subprocess, sys, tarfile, time
from concurrent.futures import ThreadPoolExecutor
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
import numpy as np, pandas as pd, nibabel as nib, matplotlib.pyplot as plt

BASE = "https://zenodo.org/records/18114231/files"
FILES = {   # name, exact size (bytes), md5 (from https://zenodo.org/api/records/18114231)
    "meta":  ("BreastDCEDL_metadata_min_crop.csv", 339901, "d58b84d0f343b4b8338cbb222e6dda08"),
    "demo":  ("BreastDCEDL_demo_data.tar.gz", 54838099, "5b7d8701309c2c3807f8d126f7ddcede"),
    "ispy1": ("BreastDCEDL_ISPY1_min_crop.tar.gz", 1252209311, "d2d2630940a861819e6a385877f45ba5"),
    "ispy2": ("BreastDCEDL_ISPY2_min_crop.tar.gz", 13634158199, "2f1a00310bd00d344664319fb7bd679c"),
    "duke":  ("BreastDCEDL_DUKE_min_crop.tar.gz", 8643461864, "16e6c752b9679138e88c188d7a01895f"),
}
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

def download(key):
    name, size, md5 = FILES[key]
    out = os.path.join(OUT, name)
    if os.path.exists(out) and os.path.getsize(out) == size and md5sum(out) == md5:
        print(f"{name}: already here, md5 OK"); return out
    url, step, t = f"{BASE}/{name}?download=1", -(-size // PARTS), time.time()
    def part(i):
        a, b = i * step, min(size, (i + 1) * step) - 1
        p = f"{out}.part{i}"
        for attempt in range(60):          # resume until the part has exactly the right size
            have = os.path.getsize(p) if os.path.exists(p) else 0
            if have >= b - a + 1:
                return
            subprocess.run(["curl", "-sL", "--fail", "-m", "600", "-r", f"{a + have}-{b}", url], stdout=open(p, "ab"))
            time.sleep(min(30, 2 * attempt))
        raise RuntimeError(f"part {i} did not complete")
    with ThreadPoolExecutor(PARTS) as ex:
        list(ex.map(part, range(PARTS)))
    with open(out, "wb") as f:
        for i in range(PARTS):
            with open(f"{out}.part{i}", "rb") as p:
                while (b := p.read(1 << 24)):
                    f.write(b)
            os.remove(f"{out}.part{i}")
    dt = time.time() - t
    ok = md5sum(out) == md5
    print(f"{name}: {size / 1e6:.1f} MB in {dt:.0f} s = {size / 1e6 / dt:.2f} MB/s, md5 {'OK' if ok else 'MISMATCH'}")
    assert ok, "md5 mismatch: delete the file and rerun"
    return out

META = pd.read_csv(download("meta")).set_index("pid", drop=False)
ARCHIVE = download(WHAT)

## The archive: one 3D NIfTI file per DCE phase, plus a tumour mask (I-SPY) per patient
Only the three phases used for the model are shipped: pre-contrast, early and late post-contrast. Their indices
are the CSV columns `pre`, `post_early` and `post_late`.

In [ ]:
import re
DATA = os.path.join(OUT, "extracted"); os.makedirs(DATA, exist_ok=True)
with tarfile.open(ARCHIVE) as tf:
    members = [m for m in tf.getmembers() if m.name.endswith(".nii.gz")]
    if WHAT == "demo" or not os.listdir(DATA):
        tf.extractall(DATA, members=members if WHAT == "demo" else members[:60])   # full cohorts: a few patients
rx = re.compile(r"^(?P<pid>.+?)_(?P<ds>spy1|spy2|duke)(?:_vis\d)?(?:_dce)?_a(?:qc|cq)_?(?P<k>\d+)\.nii\.gz$")
rows = []
for root, _, files in os.walk(DATA):
    for f in files:
        m = rx.match(f)
        if m: rows.append({"pid": m["pid"], "cohort": m["ds"], "phase": int(m["k"]), "file": os.path.join(root, f)})
        elif f.endswith("_mask.nii.gz"): rows.append({"pid": f.split("_spy")[0], "cohort": "mask", "phase": -1, "file": os.path.join(root, f)})
files = pd.DataFrame(rows)
print(f"{len(members)} NIfTI files in the archive; extracted {len(files)}")
display(files[files.phase >= 0].groupby(["cohort", "pid"]).phase.apply(sorted).to_frame("phases shipped"))
cols = ["dataset", "pCR", "HR", "HER2", "age", "n_xy", "n_z", "xy_spacing", "slice_thick", "pre", "post_early", "post_late",
        "mask_start", "mask_end", "sraw", "eraw", "scol", "ecol", "test"]
display(META.loc[[p for p in files.pid.unique() if p in META.index], cols])

## One patient: shapes, raw values, real voxel size
- Arrays are `(slices, 256, 256)` float64 with raw scanner values. Nothing is normalised.
- The NIfTI header spacing says 1 mm, which is wrong. The real in-plane pixel size (`xy_spacing`) and slice
  thickness (`slice_thick`) come from the CSV.
- The `test` column is the split: 0 train, 2 validation, 1 test.

In [ ]:
pids = [p for p in files[files.cohort.isin(["spy1", "spy2"])].pid.unique()] or list(files.pid.unique())
PID = PID or pids[0]
r = META.loc[PID]
ph = files[(files.pid == PID) & (files.phase >= 0)].set_index("phase").file
use = [int(r.pre), int(r.post_early), int(r.post_late)]
use = [k if k in ph.index else sorted(ph.index)[min(i, len(ph) - 1)] for i, k in enumerate(use)]
vols = [nib.load(ph[k]).get_fdata(dtype=np.float32) for k in use]
mf = files[(files.pid == PID) & (files.phase == -1)].file
mask = nib.load(mf.iloc[0]).get_fdata().astype(np.uint8) if len(mf) else None
img = nib.load(ph[use[0]])
print(f"{PID}: pCR = {r.pCR}, split = {({0: 'train', 1: 'test', 2: 'val'})[int(r.test)]}")
print("shape (slices, rows, cols):", vols[0].shape, "| dtype on disk:", img.header.get_data_dtype(),
      "| header spacing:", img.header.get_zooms(), "<- wrong; CSV:", r.xy_spacing, "mm in-plane,", r.slice_thick, "mm slices")
for name, v in zip(("pre", "early", "late"), vols):
    print(f"  {name:5s} min {v.min():8.1f}  median {np.median(v):8.1f}  99.5th pct {np.percentile(v, 99.5):8.1f}  max {v.max():8.1f}")
if mask is not None:
    z = np.flatnonzero(mask.any((1, 2)))
    z0, z1 = z.min(), z.max() + 1
    yy, xx = np.nonzero(mask.any(0))
    box = (yy.min(), yy.max() + 1, xx.min(), xx.max() + 1)
else:   # Duke: boxes only. The CSV box is in the original n_xy image (see challenges.md); fine when n_xy == 256
    z0, z1, box = int(r.mask_start), int(r.mask_end), (int(r.sraw), int(r.eraw), int(r.scol), int(r.ecol))
print(f"tumour: slices {z0}-{z1} of {vols[0].shape[0]}, box rows {box[0]}-{box[1]} cols {box[2]}-{box[3]}")

## The 3D volume is a stack of 2D slices. The tumour spans a range of them (red titles).

In [ ]:
zs = np.unique(np.linspace(0, vols[1].shape[0] - 1, 8).round().astype(int))
fig, ax = plt.subplots(1, len(zs), figsize=(2.2 * len(zs), 2.6))
for a, z in zip(ax, zs):
    a.imshow(vols[1][z], cmap="gray", vmax=np.percentile(vols[1], 99.5)); a.axis("off")
    a.set_title(f"slice {z}", color="red" if z0 <= z < z1 else "black", fontsize=8)
plt.suptitle("early post-contrast volume"); plt.tight_layout(); plt.show()

zm = (z0 + z1) // 2
fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, v, n in zip(ax, vols, ("pre-contrast", "early post", "late post")):
    a.imshow(v[zm], cmap="gray", vmax=np.percentile(vols[1], 99.5)); a.set_title(n); a.axis("off")
    if mask is not None: a.contour(mask[zm], levels=[0.5], colors="r", linewidths=0.6)
sel = mask[z0:z1] > 0 if mask is not None else np.s_[:, box[0]:box[1], box[2]:box[3]]
curve = [v[z0:z1][sel].mean() for v in vols]
ax[3].plot(["pre", "early", "late"], curve, "o-"); ax[3].set_title("mean tumour intensity (DCE curve)")
plt.tight_layout(); plt.show()
print("enhancement: early/pre = %.2f, late/early = %.2f (<1 washout, ~1 plateau, >1 persistent)" % (curve[1] / curve[0], curve[2] / curve[1]))

## 3D -> 2D RGB, as the ViT pipeline does it, and the saturation problem
The original (v1) pipeline puts R = pre, G = early and B = late. It clips all three phases **jointly** at the
volume's 0.5 / 99.5 percentiles and stores the result as uint8. Each tumour slice becomes one 2D RGB image, which
is cropped to 240 px around the tumour box (224 px at the network input).

Enhancing tumour is among the brightest tissue in a tumour-centred crop, so a large share of it sits **above**
the 99.5th percentile and is stored as 255. Washout vs persistent enhancement (G vs B) and internal texture are
then lost. On the demo archive, 48-96% of the I-SPY1 tumour-mask voxels were clipped this way.

In [ ]:
stack = np.stack(vols, -1)
lo, hi = np.percentile(stack.reshape(-1)[::4], [0.5, 99.5])
rgb = np.clip((stack - lo) / (hi - lo), 0, 1)
tum = stack[z0:z1][sel] if mask is not None else stack[z0:z1, box[0]:box[1], box[2]:box[3]].reshape(-1, 3)
print("tumour voxels clipped at the v1 ceiling: pre %.0f%%, early %.0f%%, late %.0f%%" % tuple(100 * (tum > hi).mean(0)))
cy, cx = (box[0] + box[1]) // 2, (box[2] + box[3]) // 2
def win(a, side):
    y0 = int(np.clip(cy - side // 2, 0, a.shape[0] - side)); x0 = int(np.clip(cx - side // 2, 0, a.shape[1] - side))
    return a[y0:y0 + side, x0:x0 + side]
side_t = int(np.clip(1.6 * max(box[1] - box[0], box[3] - box[2]), 64, 256))
p999 = np.percentile(stack.reshape(-1)[::4], 99.9)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
ax[0].imshow(rgb[zm]); ax[0].set_title("RGB = pre/early/late, v1 clip")
ax[1].imshow(win(rgb[zm], 224)); ax[1].set_title("224 px around the tumour (v1 input)")
ax[2].imshow(win(rgb[zm], side_t)); ax[2].set_title(f"tumour zoom ({side_t} px -> 224)")
ax[3].imshow(win(np.clip((stack[zm] - lo) / (p999 - lo), 0, 1), side_t)); ax[3].set_title("zoom, 99.9th pct ceiling")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print(f"{z1 - z0} tumour slices -> {z1 - z0} 2D RGB images for this patient; the patient's label (pCR) is shared by all of them")